# Group Lab 2: Statistics, Regression, and Uncertainty

        **Week:** Week 8

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Fit a simple regression.
- Calculate residuals.
- Visualize uncertainty.
- Interpret model limitations.

        ## Earth and environmental motivation

        Regression can describe relationships, but residuals and uncertainty decide how carefully we should interpret them.

        ## Dataset

        Weather plus streamflow, or water quality

        ## Python concepts used

        - Correlation
- Regression
- Residuals
- RMSE
- Uncertainty

## Group Lab 1 Debrief and Collaborative Debugging (First 20 Minutes)

Open the debrief card from Group Lab 1. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. The class will investigate one open problem one check at a time.

- 0-3 min: review the issue board.
- 3-11 min: student reports.
- 11-18 min: collaborative debugging.
- 18-20 min: record one reusable lesson and connect it to today's Lab.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Group roles

- Module A: data filtering and quality control.
- Module B: regression and residual analysis.
- Module C: uncertainty visualization and interpretation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from earthcourse.stats import linear_regression_summary

weather = pd.read_csv(PROCESSED_DIR / "iowa_city_weather_daily.csv", parse_dates=["date"])
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
merged = weather.merge(stream[["date", "discharge_cfs"]], on="date", how="inner")
sample = merged.dropna(subset=["precipitation_mm", "discharge_cfs"])
summary = linear_regression_summary(sample["precipitation_mm"], sample["discharge_cfs"])
print(summary)

In [ ]:
slope = summary["slope"]
intercept = summary["intercept"]
sample["predicted"] = intercept + slope * sample["precipitation_mm"]
sample["residual"] = sample["discharge_cfs"] - sample["predicted"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sample["precipitation_mm"], sample["discharge_cfs"], s=10, alpha=0.35, label="Daily data")
ax.scatter(sample["precipitation_mm"], sample["predicted"], s=6, alpha=0.25, label="Linear fit")
ax.set_xlabel("Daily precipitation (mm)")
ax.set_ylabel("Daily discharge (cfs)")
ax.set_title("Precipitation and streamflow relationship")
ax.legend()
fig.tight_layout()
plt.show()

## Guided coding: a fairer comparison, 7-day rain vs log flow

Same-day rain explains little because rivers integrate rainfall over days,
and discharge varies over orders of magnitude. Two standard fixes: sum the
rain over a trailing 7-day window, and take log10 of discharge. Compare the
r-squared values before and after.

In [ ]:
import numpy as np

sample = sample.sort_values("date").set_index("date")
sample["precip_7day_mm"] = sample["precipitation_mm"].rolling(
    "7D", min_periods=7
).sum()
sample["log10_discharge"] = np.log10(sample["discharge_cfs"])
sample = sample.reset_index().dropna(
    subset=["precip_7day_mm", "log10_discharge"]
)

summary_daily = linear_regression_summary(sample["precipitation_mm"], sample["discharge_cfs"])
summary_7day = linear_regression_summary(sample["precip_7day_mm"], sample["log10_discharge"])
print(f"Daily rain vs flow:       r-squared = {summary_daily['r_squared']:.3f}")
print(f"7-day rain vs log10 flow: r-squared = {summary_7day['r_squared']:.3f}")

## Guided coding: residual diagnostics (Module B)

A regression is judged by its residuals. Structure in the left panel
(curvature, funnels, stripes) means the model misses something systematic;
the right panel shows whether errors are roughly symmetric.

In [ ]:
fitted = summary_7day["intercept"] + summary_7day["slope"] * sample["precip_7day_mm"]
residuals = sample["log10_discharge"] - fitted

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(fitted, residuals, s=6, alpha=0.3)
ax1.axhline(0, color="firebrick", linewidth=1)
ax1.set_xlabel("Fitted log10 discharge")
ax1.set_ylabel("Residual")
ax1.set_title("Residuals vs fitted values")

ax2.hist(residuals, bins=40, color="slategray", edgecolor="white")
ax2.set_xlabel("Residual")
ax2.set_ylabel("Count")
ax2.set_title("Residual distribution")
fig.tight_layout()
plt.show()

## Guided coding: does the model hold up on unseen years?

Fitting and judging a model on the same data flatters it. Split by time,
fit on the earlier years, and score both periods.

In [ ]:
from earthcourse.stats import rmse, train_test_split_by_time

train, test = train_test_split_by_time(sample, "date", "2024-01-01")
fit = linear_regression_summary(train["precip_7day_mm"], train["log10_discharge"])
train_pred = fit["intercept"] + fit["slope"] * train["precip_7day_mm"]
test_pred = fit["intercept"] + fit["slope"] * test["precip_7day_mm"]

train_years = f"{train['date'].dt.year.min()}-{train['date'].dt.year.max()}"
test_years = f"{test['date'].dt.year.min()}-{test['date'].dt.year.max()}"
print(f"Train {train_years}: RMSE = {rmse(train['log10_discharge'], train_pred):.3f} log10 units")
print(f"Test  {test_years}: RMSE = {rmse(test['log10_discharge'], test_pred):.3f} log10 units")

## Guided coding: correlation is not the full story (Module A)

A correlation matrix is a quick screen, but high correlation can reflect
shared physics (temperature and dew point), shared seasonality, or
coincidence. Use it to choose candidate pairs, never to claim cause.

In [ ]:
variables = ["temp_mean_c", "dewpoint_mean_c", "relative_humidity_mean",
             "wind_speed_mean_mps", "precipitation_mm", "discharge_cfs"]
print(sample[variables].corr().round(2))

## Try it yourself

Plot residuals through time and discuss whether the linear model misses seasonal behavior.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Choose a trailing precipitation window of 3, 7, or 14 calendar days. Fit on dates before 2024, evaluate on 2024-2025, and make a residual plot for the test period. Report the window, train RMSE, test RMSE, and one limitation.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Repeat the 7-day regression using summer months only (June to August).
   Does the slope change, and what physical process could explain that?
2. Find the five days with the largest absolute residuals. Look up their
   dates and propose one hypothesis for each (snowmelt, upstream release,
   data problem).
3. Water-quality option: the processed water-quality file contains real
   historical samples for this site (most collected before 2012). Plot the
   seasonal cycle of nitrate (`characteristic_name == "Nitrate"` and
   `result_unit == "mg/L as N"`, value by month) and connect the timing to fertilizer application in Iowa.

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Regression model
- [ ] Residual plot
- [ ] Uncertainty discussion
- [ ] Role summary

        ## Short reflection

        What would make a regression result scientifically misleading?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
